# Задача 5. Матрица переходов для ДНК 1-го порядка

**Цель:** построить марковскую модель 1-го порядка из реальных данных.

In [1]:
import numpy as np
from Bio import SeqIO

В этой задаче будем строить матрицу переходов P — это таблица 4×4 где: строки = текущий нуклеотид (откуда идем), столбцы = следующий нуклеотид (куда идем)

In [5]:
record = next(SeqIO.parse("GCA_029856635.1_ASM2985663v1_genomic.fna", "fasta")) #работаю с асгардархеей 
seq = str(record.seq).upper()
seq = seq[:10000] #берем 10000 нуклеотидов 

#проверка что последовательность успешо записалась для дальнейшего исследования
print(f"Заголовок: {record.id}")
print(f"Длина: {len(seq)}")
print(f"Первые 50: {seq[:50]}")

Заголовок: JAHKLH010000001.1
Длина: 10000
Первые 50: CATGAAGCTAGTTATATTGTGATTAAGCGCGGTGAGAGTATGCAGGCTAC


In [7]:
nucleotides = ['A', 'C', 'G', 'T']

# Создаём словарь для всех 16 пар, изначально все нули
dinuc_counts = {}
for n1 in nucleotides:
    for n2 in nucleotides:
        dinuc_counts[n1 + n2] = 0

# Считаем пары
for i in range(len(seq) - 1):
    pair = seq[i] + seq[i+1]
    if pair in dinuc_counts: 
        dinuc_counts[pair] += 1

In [8]:
count_matrix = np.zeros((4, 4))

for i, n1 in enumerate(nucleotides):
    for j, n2 in enumerate(nucleotides):
        count_matrix[i, j] = dinuc_counts[n1 + n2]

In [10]:
row_sums = count_matrix.sum(axis=1, keepdims=True)
P = count_matrix / row_sums 

print("Матрица переходов P")
print(f"{'':6}", end="")
for n in nucleotides:
    print(f"  {n:>8}", end="")
print()
for i, n in enumerate(nucleotides):
    print(f"  {n}:  ", end="")
    for j in range(4):
        print(f"  {P[i,j]:.6f}", end="")
    print()

Матрица переходов P
               A         C         G         T
  A:    0.371528  0.122790  0.196654  0.309028
  C:    0.313514  0.173574  0.156757  0.356156
  G:    0.355844  0.199481  0.187532  0.257143
  T:    0.242209  0.185745  0.209812  0.362234


In [11]:
print("Проверка сумм строк")
for i, n in enumerate(nucleotides):
    print(f"  Строка {n}: {P[i].sum():.10f}")

Проверка сумм строк
  Строка A: 1.0000000000
  Строка C: 1.0000000000
  Строка G: 1.0000000000
  Строка T: 1.0000000000


## Нахождение стационарного распределения π

**Исходное уравнение:**

$$\pi = \pi P$$

**Преобразование:**

$$\pi - \pi P = 0 \implies \pi(I - P) = 0$$

Транспонируем чтобы привести к стандартному виду $Ax = b$:

$$(I - P)^T \pi^T = 0$$

Только проблема в том, что система вырождена, так как уравнения не независимы, следовательно решений бесконечно много.

Для того, чтобы получить распределение нужно заменить последнее уравнение на условие нормировки:

$$\pi_A + \pi_C + \pi_G + \pi_T = 1$$

**Итоговая система:**

$$\begin{pmatrix} (P^T - I)_1 \\ (P^T - I)_2 \\ (P^T - I)_3 \\ 1 \quad 1 \quad 1 \quad 1 \end{pmatrix} \pi = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

Решаем методом Гаусса через `np.linalg.solve(A, b)`.

In [14]:
A = (P.T - np.eye(4)) #матрица системы уравнений
A[-1] = 1.0

b = np.zeros(4) #правая часть
b[-1] = 1.0

# Решаем систему
pi = np.linalg.solve(A, b)

print("Стационарное распределение π")
for i, n in enumerate(nucleotides):
    print(f"  π({n}) = {pi[i]:.6f}")
print(f"  Сумма π = {pi.sum():.4f}")

Стационарное распределение π
  π(A) = 0.316938
  π(C) = 0.166411
  π(G) = 0.192523
  π(T) = 0.324127
  Сумма π = 1.0000


In [22]:
#сравнение стационарного распределения с наблюдаемыми частотами отдельных нуклеотидов
total = sum(seq.count(n) for n in nucleotides)  # не считаем N
observed = {n: seq.count(n) / total for n in nucleotides}

print("Наблюдаемые частоты нуклеотидов")
for n in nucleotides:
    print(f"  f({n}) = {observed[n]:.6f}")

print("\nСравнение π с наблюдаемыми частотами")
print(f"{'Нуклеотид':>12} | {'π (модель)':>12} | {'наблюдаемое':>12}") 
print("-" * 55)
for i, n in enumerate(nucleotides):
    print(f"{n:>12} | {pi[i]:>12.6f} | {observed[n]:>12.6f}")

Наблюдаемые частоты нуклеотидов
  f(A) = 0.316900
  f(C) = 0.166500
  f(G) = 0.192500
  f(T) = 0.324100

Сравнение π с наблюдаемыми частотами
   Нуклеотид |   π (модель) |  наблюдаемое
-------------------------------------------------------
           A |     0.316938 |     0.316900
           C |     0.166411 |     0.166500
           G |     0.192523 |     0.192500
           T |     0.324127 |     0.324100


Видим, что значения стационарного распределения практически равны наблюдаемымым частотама отдельных нуклеотидов